# Page 60: stripe-angle correction test

横帯の角度を推定し、反射padding後に水平化してからpystripeを適用し、逆回転後にFFCとmorphologyを行います。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import numpy as np
import pystripe
from scipy.ndimage import gaussian_filter, gaussian_filter1d, rotate
import tifffile as tiff

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root containing src/ was not found.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from preprocess import flatfield_like_correction, subtract_background_morphology

INPUT_TIF = PROJECT_ROOT / 'data' / '042926_MAY08R_FOS_1_retake_c.tif'
CURRENT_OUTPUT = PROJECT_ROOT / 'outputs' / '042926_MAY08R_FOS_1_retake_c_uint16_scale10000' / '042926_MAY08R_FOS_1_retake_c_060.tif'
PYSTRIPE_FIRST_OUTPUT = PROJECT_ROOT / 'outputs' / 'preprocess_order_test' / '042926_MAY08R_FOS_1_retake_c_060_pystripe_first.tif'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'preprocess_angle_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANGLE_OUTPUT = OUTPUT_DIR / '042926_MAY08R_FOS_1_retake_c_060_angle_corrected.tif'
COMPARISON_PNG = OUTPUT_DIR / '042926_MAY08R_FOS_1_retake_c_060_angle_comparison.png'

## Estimate the stripe angle

画像を候補角度で回転し、局所的な細胞を抑える行中央値プロファイルの高周波成分が最も強く揃う角度を選びます。

In [ ]:
current = tiff.imread(str(CURRENT_OUTPUT)).astype(np.float32)
angle_roi = current[1400:3000:4, 400:1500:4]
angle_highpass = angle_roi - gaussian_filter(angle_roi, sigma=(7.5, 7.5))
angle_highpass *= np.outer(np.hanning(angle_highpass.shape[0]), np.hanning(angle_highpass.shape[1]))

def alignment_score(angle):
    aligned = rotate(
        angle_highpass, angle=angle, reshape=False, order=1,
        mode='constant', cval=0, prefilter=False
    )
    profile = np.median(aligned[30:-30, 30:-30], axis=1)
    return np.std(profile)

candidate_angles = np.arange(-3.0, 3.0001, 0.025)
scores = np.array([alignment_score(angle) for angle in candidate_angles])
stripe_angle = float(candidate_angles[np.argmax(scores)])
print('Estimated correction angle: {:.3f} degrees'.format(stripe_angle))

In [ ]:
with tiff.TiffFile(str(INPUT_TIF)) as tif:
    raw = tif.pages[59].asarray().astype(np.float32)

offset = np.percentile(raw, 0.05)
raw0 = raw - offset
raw0[raw0 < 0] = 0

pad = 128
padded = np.pad(raw0, ((pad, pad), (pad, pad)), mode='reflect')
aligned = rotate(
    padded, angle=stripe_angle, reshape=False, order=1,
    mode='reflect', prefilter=False
)
aligned_destriped = pystripe.filter_streaks(
    aligned, sigma=(128, 256), level=7, wavelet='db2'
).astype(np.float32)
restored = rotate(
    aligned_destriped, angle=-stripe_angle, reshape=False, order=1,
    mode='reflect', prefilter=False
)
restored = restored[pad:-pad, pad:-pad]

ffc, field = flatfield_like_correction(
    restored, sigma=120, reference_level=100, max_gain=3.0
)
preprocessed, background = subtract_background_morphology(ffc, radius=20)
output_uint16 = np.clip(preprocessed * 100.0, 0, 65535).astype(np.uint16)
tiff.imwrite(str(ANGLE_OUTPUT), output_uint16)
print('Saved:', ANGLE_OUTPUT)

In [ ]:
def horizontal_band_metric(image):
    roi = np.asarray(image[1000:3500, 300:1900], dtype=np.float32)
    row_profile = np.median(roi, axis=1)
    residual = row_profile - gaussian_filter1d(row_profile, sigma=30)
    robust_range = np.percentile(roi, 99) - np.percentile(roi, 1)
    return 100.0 * np.std(residual) / robust_range

def angle_aligned_band_metric(image, correction_angle):
    aligned_for_measurement = rotate(
        np.asarray(image, dtype=np.float32), angle=correction_angle,
        reshape=False, order=1, mode='reflect', prefilter=False
    )
    return horizontal_band_metric(aligned_for_measurement)

current = tiff.imread(str(CURRENT_OUTPUT))
pystripe_first = tiff.imread(str(PYSTRIPE_FIRST_OUTPUT))
angle_corrected = tiff.imread(str(ANGLE_OUTPUT))

for name, image in (
    ('Current: FFC -> pystripe', current),
    ('Pystripe -> FFC', pystripe_first),
    ('Angle correction -> pystripe -> restore -> FFC', angle_corrected),
):
    print(
        '{}: original={:.4f}%, angle-aligned={:.4f}%'.format(
            name, horizontal_band_metric(image),
            angle_aligned_band_metric(image, stripe_angle)
        )
    )

In [ ]:
norm = PowerNorm(gamma=0.7, vmin=0, vmax=5500)
images = (current, pystripe_first, angle_corrected)
titles = (
    'Current: FFC -> pystripe',
    'Pystripe -> FFC',
    'Angle {:.3f}° -> pystripe -> restore -> FFC'.format(stripe_angle),
)
fig, axes = plt.subplots(1, 3, figsize=(15, 10), dpi=160, facecolor='black')
for ax, image, title in zip(axes, images, titles):
    ax.imshow(image, cmap='gray', norm=norm)
    ax.set_title('{}\nmetric = {:.4f}%'.format(title, horizontal_band_metric(image)), color='white', fontsize=10)
    ax.axis('off')
plt.tight_layout()
fig.savefig(str(COMPARISON_PNG), facecolor='black', bbox_inches='tight')
plt.show()
print('Saved:', COMPARISON_PNG)